# 20 — Malignant-state composition and program scoring

This notebook is the repository-numbered equivalent of the historical executed notebook:

`19_malignant_state_composition_program_scoring_HVG8000_r0_6.ipynb`

The malignant-state definitions, statistical tests, program-marker panels, Scanpy gene-scoring procedure, figures, and output names are preserved from that historical analysis.

## Historical malignant-state definitions

The five analysis states are defined directly from the canonical Leiden clusters:

- cluster `0` → `OPC_NPC_like_malignant`
- cluster `4` → `hypoxia_stress_malignant`
- cluster `7` → `AC_like_reactive_malignant`
- cluster `11` → `NPC_neuronal_like_malignant`
- cluster `2` → `cycling_malignant_candidate`

All other clusters are labeled `not_malignant_state`.

## Canonical input policy

This notebook requires the canonical Notebook-17 final annotation object:

`GBM_core_4datasets_final_annotation_CNV_consensus.h5ad`

The compatibility alias used by the historical notebook is not used as the computational source. Before analysis, exact canonical cell/gene, CopyKAT, compartment, cluster, and marker-panel checkpoints are verified.

## Memory-safety policy

The historical notebook loaded the complete full-gene H5AD, including both expression data and the 739,913,124-entry counts layer, into RAM. This implementation avoids that pattern.

- `/obs`, `/var`, and `X_umap` are read directly with h5py/read_elem.
- Only the 161,884 malignant-state-analysis cells are extracted from `/layers/counts`.
- The extraction is streamed by CSR row blocks to a temporary counts-only H5AD on local scratch.
- That temporary object contains one sparse matrix only; it has no duplicate counts layer.
- Scanpy `normalize_total`, `log1p`, and `score_genes` are then run on that exact malignant-cell/full-gene matrix, preserving historical scoring semantics.
- The final full H5AD is created by sequentially copying the canonical Notebook-17 input and replacing `/obs` only.
- The complete counts layer is never materialized in memory.

The temporary scoring H5AD is deleted after successful completion.

## Sample-identifier compatibility

The historical sample-level score display used the literal identifier `GSM4202144_PJ052`. Because repository metadata harmonization can retain the same biological sample under a longer or shortened identifier, the optional sample-level sentinel is resolved using the stable `PJ052` token across sample and patient identifiers. The authoritative hard score checks remain the state-level historical score sentinels across all 22,525 AC-like cells plus the exact `(375, 20)` sample-score table shape.

## Power-safe normalization/scoring revision

The full malignant-cell CSR contains 383,992,077 stored values. Calling
`sc.pp.normalize_total()` on that complete in-memory matrix can require a
second per-nonzero scaling array and exceed the available RAM.

This revision keeps the exact historical Scanpy operations but applies
`normalize_total(target_sum=1e4)` followed by `log1p` to independent 512-cell
CSR row blocks. The normalized blocks are written to a separate float32
scratch H5AD. Progress is committed after each block, so a power interruption
can resume from the last completed row block instead of restarting.

After normalization is complete, the raw-count scratch is removed and only
the normalized/log1p matrix is loaded for the historical `sc.tl.score_genes`
calls. The authoritative historical score sentinels remain unchanged.


In [1]:
# =========================
# 0. Imports and paths
# =========================
from pathlib import Path
import os
import re
import shutil
import warnings
import gc

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad
import h5py
import matplotlib.pyplot as plt

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

try:
    from anndata.io import read_elem, write_elem
except Exception:
    from anndata._io.specs import read_elem, write_elem


sc.settings.verbosity = 2


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
    / "final_malignant_state_analysis"
)

for directory in [
    PROCESSED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


INPUT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_final_annotation_CNV_consensus.h5ad"
)

OUTPUT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
)

WRITE_UPDATED_H5AD = True

CLUSTER_KEY = "leiden_scvi_main"
FINAL_COMPARTMENT_COL = "final_compartment_major"
FINAL_ANNOTATION_COL = "final_annotation_refined"
COPYKAT_COL = "copykat_pred_clean"

CONDITION_COL = "condition"

SAMPLE_COL_CANDIDATES = [
    "sample_id",
    "sample",
    "orig.ident",
    "sample_name",
]

PATIENT_COL_CANDIDATES = [
    "patient_id",
    "patient",
    "patient_name",
]

DATASET_COL_CANDIDATES = [
    "core_dataset",
    "dataset_id",
    "dataset",
]

COUNTS_LAYER_CANDIDATES = [
    "counts",
    "raw_counts",
]

RANDOM_STATE = 42


def choose_scratch_root():
    candidates = []

    env_scratch = os.environ.get(
        "GLIOMA_SCRATCH_ROOT"
    )

    if env_scratch:
        candidates.append(
            Path(
                env_scratch
            )
        )

    if os.name == "nt":
        candidates.append(
            Path(
                r"E:\glioma-cnv-scratch\main_analysis_20"
            )
        )

    candidates.append(
        PROJECT_DIR
        / "data"
        / "interim"
        / "scratch_main_analysis_20"
    )

    for candidate in candidates:
        try:
            candidate.mkdir(
                parents=True,
                exist_ok=True,
            )

            if candidate.exists():
                return candidate.resolve()

        except Exception:
            continue

    raise RuntimeError(
        "Could not create a writable scratch directory."
    )


SCRATCH_DIR = choose_scratch_root()

SCORING_COUNTS_H5AD = (
    SCRATCH_DIR
    / "20_malignant_cells_fullgene_counts_only_for_scoring.h5ad"
)

SCORING_NORM_H5AD = (
    SCRATCH_DIR
    / "20_malignant_cells_fullgene_normalized_log1p_for_scoring.h5ad"
)

# Backward-compatible name used only in cleanup messages from earlier revisions.
SCORING_H5AD = SCORING_COUNTS_H5AD


print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "INPUT_H5AD:",
    INPUT_H5AD,
)

print(
    "OUTPUT_H5AD:",
    OUTPUT_H5AD,
)

print(
    "SCRATCH_DIR:",
    SCRATCH_DIR,
)

print(
    "SCORING_COUNTS_H5AD:",
    SCORING_COUNTS_H5AD,
)

print(
    "SCORING_NORM_H5AD:",
    SCORING_NORM_H5AD,
)


if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        "Notebook 20 requires the canonical Notebook-17 final annotation H5AD:\n"
        f"{INPUT_H5AD}"
    )


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_CNV_consensus.h5ad
OUTPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
SCRATCH_DIR: E:\glioma-cnv-scratch\main_analysis_20
SCORING_COUNTS_H5AD: E:\glioma-cnv-scratch\main_analysis_20\20_malignant_cells_fullgene_counts_only_for_scoring.h5ad
SCORING_NORM_H5AD: E:\glioma-cnv-scratch\main_analysis_20\20_malignant_cells_fullgene_normalized_log1p_for_scoring.h5ad


In [2]:
# =========================
# 1. Helpers
# =========================
def write_tsv_replace(
    df,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=False,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )


def savefig_replace(
    path,
    dpi=220,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    plt.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"[SAVED FIG] {path}"
    )


def choose_first_existing_column(
    obs_or_var,
    candidates,
    required=False,
    label="column",
):
    for candidate in candidates:
        if candidate in obs_or_var.columns:
            return candidate

    if required:
        raise KeyError(
            f"Could not find required {label}. Tried: {candidates}"
        )

    return None


def normalize_label(value):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    text = (
        text
        .replace(
            "/",
            " ",
        )
        .replace(
            "-",
            " ",
        )
    )

    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text,
    )

    return re.sub(
        r"_+",
        "_",
        text,
    ).strip(
        "_"
    )


def bh_fdr(
    pvals,
):
    pvals = np.asarray(
        pvals,
        dtype=float,
    )

    q = np.full_like(
        pvals,
        np.nan,
        dtype=float,
    )

    ok = np.isfinite(
        pvals
    )

    if ok.sum() > 0:
        q[
            ok
        ] = multipletests(
            pvals[
                ok
            ],
            method="fdr_bh",
        )[
            1
        ]

    return q


def safe_mannwhitney(
    x,
    y,
):
    x = np.asarray(
        pd.Series(
            x
        ).dropna(),
        dtype=float,
    )

    y = np.asarray(
        pd.Series(
            y
        ).dropna(),
        dtype=float,
    )

    if (
        len(
            x
        ) < 1
        or len(
            y
        ) < 1
    ):
        return np.nan

    try:
        return mannwhitneyu(
            x,
            y,
            alternative="two-sided",
        ).pvalue

    except Exception:
        return np.nan


def safe_kruskal(
    groups,
):
    groups = [
        np.asarray(
            pd.Series(
                group
            ).dropna(),
            dtype=float,
        )
        for group in groups
        if len(
            pd.Series(
                group
            ).dropna()
        ) > 0
    ]

    if len(
        groups
    ) < 2:
        return np.nan

    all_values = np.concatenate(
        groups
    )

    if len(
        np.unique(
            all_values
        )
    ) <= 1:
        return np.nan

    try:
        return kruskal(
            *groups
        ).pvalue

    except Exception:
        return np.nan


def normalize_copykat_value(
    value,
):
    if pd.isna(
        value
    ):
        return "missing"

    text = str(
        value
    ).strip().lower()

    mapping = {
        "aneuploid": "aneuploid",
        "diploid": "diploid",
        "not.defined": "not_defined",
        "not_defined": "not_defined",
        "missing": "missing",
        "nan": "missing",
    }

    return mapping.get(
        text,
        text,
    )


def counts_dict(
    series,
):
    return {
        str(
            key
        ): int(
            value
        )
        for key, value
        in series.value_counts(
            dropna=False
        ).items()
    }


def copy_file_with_progress(
    src,
    dst,
    chunk_size=64 * 1024 * 1024,
):
    src = Path(
        src
    )

    dst = Path(
        dst
    )

    total = src.stat().st_size
    copied = 0
    next_report = 5

    with src.open(
        "rb"
    ) as fin, dst.open(
        "wb"
    ) as fout:
        while True:
            block = fin.read(
                chunk_size
            )

            if not block:
                break

            fout.write(
                block
            )

            copied += len(
                block
            )

            pct = int(
                copied
                * 100
                / max(
                    total,
                    1,
                )
            )

            if pct >= next_report:
                print(
                    f"  {pct}% "
                    f"({copied / 1024**3:.2f} / "
                    f"{total / 1024**3:.2f} GiB)"
                )

                next_report += 5

        fout.flush()
        os.fsync(
            fout.fileno()
        )

    shutil.copystat(
        src,
        dst,
    )


def replace_h5ad_obs_group(
    h5ad_path,
    obs_df,
):
    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "obs" in handle:
            del handle[
                "obs"
            ]

        write_elem(
            handle,
            "obs",
            obs_df,
        )

        handle.flush()


def add_h5ad_uns_elements(
    h5ad_path,
    elements,
):
    if not elements:
        return

    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "uns" not in handle:
            handle.create_group(
                "uns"
            )

        uns_group = handle[
            "uns"
        ]

        for key, value in elements.items():
            if key in uns_group:
                del uns_group[
                    key
                ]

            write_elem(
                uns_group,
                key,
                value,
            )

        handle.flush()


def validate_h5ad_obs_only(
    h5ad_path,
    expected_shape,
    expected_state_counts,
):
    with h5py.File(
        h5ad_path,
        "r",
    ) as handle:

        obs_check = read_elem(
            handle[
                "obs"
            ]
        )

        var_check = read_elem(
            handle[
                "var"
            ]
        )

        observed_shape = (
            int(
                obs_check.shape[
                    0
                ]
            ),
            int(
                var_check.shape[
                    0
                ]
            ),
        )

        if observed_shape != expected_shape:
            raise RuntimeError(
                f"Output H5AD shape mismatch: "
                f"{observed_shape} != {expected_shape}"
            )

        required = [
            "malignant_state",
            "is_malignant_state_analysis_cell",
        ]

        required.extend(
            [
                f"score_{name}"
                for name in PROGRAM_MARKERS
            ]
        )

        missing = [
            column
            for column in required
            if column not in obs_check.columns
        ]

        if missing:
            raise RuntimeError(
                "Output H5AD lacks required Notebook-20 obs columns: "
                + ", ".join(
                    missing
                )
            )

        observed_state_counts = counts_dict(
            obs_check[
                "malignant_state"
            ]
        )

        if (
            observed_state_counts
            != expected_state_counts
        ):
            raise RuntimeError(
                "Output H5AD malignant-state checkpoint failed. "
                f"Observed: {observed_state_counts}"
            )

    return observed_shape


figure_records = []
manifest_records = []

def replace_h5ad_dataframe_group(
    h5ad_path,
    group_name,
    df,
):
    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if group_name in handle:
            del handle[
                group_name
            ]

        write_elem(
            handle,
            group_name,
            df,
        )

        handle.flush()


def _decode_h5_string_array(values):
    values = np.asarray(
        values
    )

    if values.dtype.kind in {
        "S",
        "O",
    }:
        return np.asarray(
            [
                (
                    value.decode(
                        "utf-8"
                    )
                    if isinstance(
                        value,
                        (
                            bytes,
                            np.bytes_,
                        ),
                    )
                    else str(
                        value
                    )
                )
                for value in values
            ],
            dtype=object,
        )

    return values.astype(
        str
    )


def read_h5_dataframe_index(
    group,
):
    index_key = group.attrs.get(
        "_index",
        "_index",
    )

    if isinstance(
        index_key,
        (
            bytes,
            np.bytes_,
        ),
    ):
        index_key = index_key.decode(
            "utf-8"
        )

    if index_key not in group:
        raise RuntimeError(
            f"H5AD dataframe group {group.name} lacks index dataset {index_key!r}."
        )

    return _decode_h5_string_array(
        group[
            index_key
        ][:]
    )


def audit_scoring_scratch(
    h5ad_path,
    expected_obs_names,
    expected_n_vars,
    expected_nnz,
    require_float32=False,
    require_complete_normalization=False,
):
    h5ad_path = Path(
        h5ad_path
    )

    if not h5ad_path.exists():
        return {
            "valid": False,
            "reason": "missing",
        }

    try:
        with h5py.File(
            h5ad_path,
            "r",
        ) as handle:

            for required in [
                "obs",
                "var",
                "X",
            ]:
                if required not in handle:
                    return {
                        "valid": False,
                        "reason": f"missing /{required}",
                    }

            x_group = handle[
                "X"
            ]

            encoding = str(
                x_group.attrs.get(
                    "encoding-type",
                    "",
                )
            )

            if encoding != "csr_matrix":
                return {
                    "valid": False,
                    "reason": f"X encoding {encoding!r}",
                }

            for required in [
                "data",
                "indices",
                "indptr",
            ]:
                if required not in x_group:
                    return {
                        "valid": False,
                        "reason": f"missing /X/{required}",
                    }

            obs_names = read_h5_dataframe_index(
                handle[
                    "obs"
                ]
            )

            n_vars = len(
                read_h5_dataframe_index(
                    handle[
                        "var"
                    ]
                )
            )

            if len(
                obs_names
            ) != len(
                expected_obs_names
            ):
                return {
                    "valid": False,
                    "reason": "obs length mismatch",
                }

            if not np.array_equal(
                obs_names.astype(
                    str
                ),
                np.asarray(
                    expected_obs_names,
                    dtype=str,
                ),
            ):
                return {
                    "valid": False,
                    "reason": "obs order mismatch",
                }

            if n_vars != expected_n_vars:
                return {
                    "valid": False,
                    "reason": "var length mismatch",
                }

            observed_nnz = int(
                x_group[
                    "data"
                ].shape[
                    0
                ]
            )

            if observed_nnz != expected_nnz:
                return {
                    "valid": False,
                    "reason": (
                        f"nnz mismatch {observed_nnz} != {expected_nnz}"
                    ),
                }

            indptr_last = int(
                x_group[
                    "indptr"
                ][
                    -1
                ]
            )

            if indptr_last != expected_nnz:
                return {
                    "valid": False,
                    "reason": (
                        f"indptr[-1] mismatch {indptr_last} != {expected_nnz}"
                    ),
                }

            data_dtype = np.dtype(
                x_group[
                    "data"
                ].dtype
            )

            if (
                require_float32
                and data_dtype
                != np.dtype(
                    np.float32
                )
            ):
                return {
                    "valid": False,
                    "reason": f"data dtype is {data_dtype}, not float32",
                }

            normalized_rows_complete = int(
                x_group.attrs.get(
                    "normalized_rows_complete",
                    0,
                )
            )

            normalization_complete = bool(
                x_group.attrs.get(
                    "normalization_complete",
                    False,
                )
            )

            if (
                require_complete_normalization
                and (
                    normalized_rows_complete
                    != len(
                        expected_obs_names
                    )
                    or not normalization_complete
                )
            ):
                return {
                    "valid": False,
                    "reason": (
                        "normalized scratch is incomplete "
                        f"(rows={normalized_rows_complete}, "
                        f"complete={normalization_complete})"
                    ),
                    "resumable": True,
                    "normalized_rows_complete": normalized_rows_complete,
                }

            return {
                "valid": True,
                "reason": "ok",
                "data_dtype": str(
                    data_dtype
                ),
                "normalized_rows_complete": normalized_rows_complete,
                "normalization_complete": normalization_complete,
            }

    except Exception as error:
        return {
            "valid": False,
            "reason": repr(
                error
            ),
        }


In [3]:
# =========================
# 2. Historical program marker panels
# =========================
PROGRAM_MARKERS = {
    "OPC_proneural_program": [
        "PDGFRA",
        "OLIG1",
        "OLIG2",
        "PTPRZ1",
        "BCAN",
        "SOX6",
        "SOX10",
        "CSPG4",
        "NKX2-2",
    ],
    "NPC_neural_program": [
        "SOX4",
        "SOX11",
        "DCX",
        "STMN2",
        "TUBB3",
        "DLX5",
        "MYT1L",
        "ASCL1",
        "NEUROD1",
    ],
    "AC_like_reactive_program": [
        "GFAP",
        "AQP4",
        "ALDH1L1",
        "SLC1A3",
        "CLU",
        "SPARCL1",
        "APOE",
        "VIM",
    ],
    "MES_like_program": [
        "CHI3L1",
        "CD44",
        "VIM",
        "FN1",
        "SERPINE1",
        "TGFBI",
        "LGALS3",
        "CCL2",
        "ITGA5",
    ],
    "hypoxia_stress_program": [
        "CA9",
        "VEGFA",
        "HILPDA",
        "LOX",
        "BNIP3",
        "PDK1",
        "SLC2A1",
        "DDIT4",
        "NDRG1",
    ],
    "cycling_program": [
        "MKI67",
        "TOP2A",
        "PBK",
        "BIRC5",
        "UBE2C",
        "CENPF",
        "NUSAP1",
        "CENPA",
        "HMGB2",
    ],
    "myeloid_contamination_review_program": [
        "C1QA",
        "C1QB",
        "C1QC",
        "TYROBP",
        "LYZ",
        "FCN1",
        "CD74",
        "HLA-DRA",
        "CSF1R",
    ],
}


In [4]:
# =========================
# 3. Read canonical Notebook-17 metadata only
# =========================
with h5py.File(
    INPUT_H5AD,
    "r",
) as handle:

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    var = read_elem(
        handle[
            "var"
        ]
    )

    obs.index = (
        obs.index
        .astype(str)
    )

    var.index = (
        var.index
        .astype(str)
    )

    source_shape = (
        obs.shape[
            0
        ],
        var.shape[
            0
        ],
    )

    obsm_keys = list(
        handle[
            "obsm"
        ].keys()
    ) if "obsm" in handle else []

    layer_keys = list(
        handle[
            "layers"
        ].keys()
    ) if "layers" in handle else []

    if (
        "obsm"
        not in handle
        or "X_umap"
        not in handle[
            "obsm"
        ]
    ):
        raise RuntimeError(
            "Canonical Notebook-17 H5AD lacks X_umap."
        )

    source_umap = read_elem(
        handle[
            "obsm"
        ][
            "X_umap"
        ]
    )

    counts_layer = None

    for candidate in COUNTS_LAYER_CANDIDATES:
        if (
            "layers"
            in handle
            and candidate
            in handle[
                "layers"
            ]
        ):
            counts_layer = candidate
            break

    if counts_layer is None:
        raise RuntimeError(
            "Canonical Notebook-17 H5AD lacks the historical counts layer."
        )

    counts_group = handle[
        "layers"
    ][
        counts_layer
    ]

    counts_encoding = str(
        counts_group.attrs.get(
            "encoding-type",
            "",
        )
    )

    if counts_encoding != "csr_matrix":
        raise RuntimeError(
            "Memory-safe Notebook 20 currently requires a CSR counts layer. "
            f"Observed encoding: {counts_encoding}"
        )

    full_indptr = np.asarray(
        counts_group[
            "indptr"
        ][:]
    )

    counts_data_dtype = (
        counts_group[
            "data"
        ].dtype
    )

    counts_indices_dtype = (
        counts_group[
            "indices"
        ].dtype
    )


sample_col = choose_first_existing_column(
    obs,
    SAMPLE_COL_CANDIDATES,
    required=True,
    label="sample column",
)

patient_col = choose_first_existing_column(
    obs,
    PATIENT_COL_CANDIDATES,
    required=False,
    label="patient column",
)

dataset_col = choose_first_existing_column(
    obs,
    DATASET_COL_CANDIDATES,
    required=False,
    label="dataset column",
)


required_cols = [
    CLUSTER_KEY,
    FINAL_COMPARTMENT_COL,
    FINAL_ANNOTATION_COL,
    CONDITION_COL,
    COPYKAT_COL,
]

missing_cols = [
    column
    for column in required_cols
    if column not in obs.columns
]

if missing_cols:
    raise KeyError(
        f"Missing required obs columns: {missing_cols}"
    )


print(
    "source shape:",
    source_shape,
)

print(
    "sample_col:",
    sample_col,
)

print(
    "patient_col:",
    patient_col,
)

print(
    "dataset_col:",
    dataset_col,
)

print(
    "counts_layer:",
    counts_layer,
)

print(
    "layers:",
    layer_keys,
)

print(
    "obsm:",
    obsm_keys,
)


if source_shape != (
    351427,
    15176,
):
    raise RuntimeError(
        f"Canonical Notebook-17 shape checkpoint failed: {source_shape}"
    )


EXPECTED_COPYKAT_COUNTS = {
    "aneuploid": 202895,
    "diploid": 111493,
    "not_defined": 37037,
    "missing": 2,
}

observed_copykat_counts = counts_dict(
    obs[
        COPYKAT_COL
    ]
    .astype(object)
    .map(
        normalize_copykat_value
    )
)

if observed_copykat_counts != EXPECTED_COPYKAT_COUNTS:
    raise RuntimeError(
        "Canonical CopyKAT checkpoint failed. "
        f"Observed: {observed_copykat_counts}"
    )


EXPECTED_COMPARTMENT_COUNTS = {
    "review": 167178,
    "malignant_like": 123250,
    "cycling_malignant_like_candidate": 38634,
    "non_malignant_or_diploid_leaning": 22365,
}

observed_compartment_counts = counts_dict(
    obs[
        FINAL_COMPARTMENT_COL
    ]
)

if (
    observed_compartment_counts
    != EXPECTED_COMPARTMENT_COUNTS
):
    raise RuntimeError(
        "Canonical final-compartment checkpoint failed. "
        f"Observed: {observed_compartment_counts}"
    )


if (
    obs[
        FINAL_ANNOTATION_COL
    ]
    .astype(str)
    .nunique()
    != 18
):
    raise RuntimeError(
        "Canonical refined-annotation checkpoint failed."
    )


# Historical marker-panel presence must be complete.
var_name_set = set(
    var.index
)

for program, genes in PROGRAM_MARKERS.items():
    missing_program_genes = [
        gene
        for gene in genes
        if gene not in var_name_set
    ]

    if missing_program_genes:
        raise RuntimeError(
            f"Historical marker panel is incomplete for {program}: "
            f"{missing_program_genes}"
        )


print(
    "PASS: canonical Notebook-17 input checkpoints match exactly."
)


source shape: (351427, 15176)
sample_col: sample_id
patient_col: patient_id
dataset_col: core_dataset
counts_layer: counts
layers: ['counts']
obsm: ['X_scVI', 'X_umap']
PASS: canonical Notebook-17 input checkpoints match exactly.


In [5]:
# =========================
# 4. Define malignant-like states from final clusters
# =========================
CLUSTER_TO_MALIGNANT_STATE = {
    "0": "OPC_NPC_like_malignant",
    "4": "hypoxia_stress_malignant",
    "7": "AC_like_reactive_malignant",
    "11": "NPC_neuronal_like_malignant",
    "2": "cycling_malignant_candidate",
}


obs[
    CLUSTER_KEY
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
)

obs[
    "malignant_state"
] = (
    obs[
        CLUSTER_KEY
    ]
    .map(
        CLUSTER_TO_MALIGNANT_STATE
    )
    .fillna(
        "not_malignant_state"
    )
)

obs[
    "is_malignant_state_analysis_cell"
] = (
    obs[
        "malignant_state"
    ]
    .ne(
        "not_malignant_state"
    )
)

obs[
    "_final_compartment_norm"
] = (
    obs[
        FINAL_COMPARTMENT_COL
    ]
    .map(
        normalize_label
    )
)


mal_state_counts = (
    obs[
        "malignant_state"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "malignant_state"
    )
    .reset_index(
        name="n_cells"
    )
)

mal_state_counts[
    "fraction_all_cells"
] = (
    mal_state_counts[
        "n_cells"
    ]
    / source_shape[
        0
    ]
)


EXPECTED_STATE_COUNTS = {
    "not_malignant_state": 189543,
    "OPC_NPC_like_malignant": 61646,
    "cycling_malignant_candidate": 38634,
    "hypoxia_stress_malignant": 32896,
    "AC_like_reactive_malignant": 22525,
    "NPC_neuronal_like_malignant": 6183,
}

observed_state_counts = {
    str(
        row[
            "malignant_state"
        ]
    ): int(
        row[
            "n_cells"
        ]
    )
    for _, row
    in mal_state_counts.iterrows()
}

if (
    observed_state_counts
    != EXPECTED_STATE_COUNTS
):
    raise RuntimeError(
        "Historical malignant-state count checkpoint failed. "
        f"Observed: {observed_state_counts}"
    )


STATE_COUNTS_TSV = (
    METADATA_DIR
    / "malignant_state_cell_counts_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    mal_state_counts,
    STATE_COUNTS_TSV,
)

manifest_records.append(
    {
        "file": str(
            STATE_COUNTS_TSV
        ),
        "description": "Cell counts for malignant-state labels",
    }
)

display(
    mal_state_counts
)

print(
    "PASS: historical malignant-state counts reproduced exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_cell_counts_HVG8000_r0_6.tsv shape=(6, 3)


,malignant_state,n_cells,fraction_all_cells
0,not_malignant_state,189543,0.539352
1,OPC_NPC_like_malignant,61646,0.175416
2,cycling_malignant_candidate,38634,0.109935
3,hypoxia_stress_malignant,32896,0.093607
4,AC_like_reactive_malignant,22525,0.064096
5,NPC_neuronal_like_malignant,6183,0.017594


PASS: historical malignant-state counts reproduced exactly.


In [6]:
# =========================
# 5. Input readiness without materializing counts
# =========================
# Reproduce the historical 1000 x 1000 count sample directly from CSR storage.
with h5py.File(
    INPUT_H5AD,
    "r",
) as handle:

    counts_group = handle[
        "layers"
    ][
        counts_layer
    ]

    n_sample_rows = min(
        1000,
        source_shape[
            0
        ],
    )

    start_ptr = int(
        full_indptr[
            0
        ]
    )

    end_ptr = int(
        full_indptr[
            n_sample_rows
        ]
    )

    sample_data = np.asarray(
        counts_group[
            "data"
        ][
            start_ptr:end_ptr
        ]
    )

    sample_indices = np.asarray(
        counts_group[
            "indices"
        ][
            start_ptr:end_ptr
        ]
    )

    sample_indptr = (
        full_indptr[
            : n_sample_rows + 1
        ]
        - start_ptr
    )

    sample_csr = sp.csr_matrix(
        (
            sample_data,
            sample_indices,
            sample_indptr,
        ),
        shape=(
            n_sample_rows,
            source_shape[
                1
            ],
        ),
    )

    sample_csr = (
        sample_csr[
            :,
            : min(
                1000,
                source_shape[
                    1
                ],
            ),
        ]
    )

    vals = (
        sample_csr.data
    )

    vals = vals[
        np.isfinite(
            vals
        )
    ]


readiness = pd.DataFrame(
    [
        {
            "item": "n_cells",
            "value": source_shape[
                0
            ],
        },
        {
            "item": "n_genes",
            "value": source_shape[
                1
            ],
        },
        {
            "item": "input_h5ad_exists",
            "value": INPUT_H5AD.exists(),
        },
        {
            "item": "cluster_key",
            "value": CLUSTER_KEY,
        },
        {
            "item": "final_compartment_col",
            "value": FINAL_COMPARTMENT_COL,
        },
        {
            "item": "final_annotation_col",
            "value": FINAL_ANNOTATION_COL,
        },
        {
            "item": "condition_col",
            "value": CONDITION_COL,
        },
        {
            "item": "sample_col",
            "value": sample_col,
        },
        {
            "item": "patient_col",
            "value": (
                patient_col
                if patient_col
                else "NA"
            ),
        },
        {
            "item": "dataset_col",
            "value": (
                dataset_col
                if dataset_col
                else "NA"
            ),
        },
        {
            "item": "counts_source_for_scoring",
            "value": counts_layer,
        },
        {
            "item": "counts_min_sample",
            "value": (
                float(
                    vals.min()
                )
                if len(
                    vals
                )
                else np.nan
            ),
        },
        {
            "item": "counts_max_sample",
            "value": (
                float(
                    vals.max()
                )
                if len(
                    vals
                )
                else np.nan
            ),
        },
        {
            "item": "counts_integer_like_sample",
            "value": (
                bool(
                    np.allclose(
                        vals,
                        np.round(
                            vals
                        ),
                    )
                )
                if len(
                    vals
                )
                else np.nan
            ),
        },
        {
            "item": "has_X_umap",
            "value": (
                "X_umap"
                in obsm_keys
            ),
        },
    ]
)


READINESS_TSV = (
    METADATA_DIR
    / "malignant_state_readiness_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    readiness,
    READINESS_TSV,
)

manifest_records.append(
    {
        "file": str(
            READINESS_TSV
        ),
        "description": "Input readiness report",
    }
)


if readiness.shape != (
    15,
    2,
):
    raise RuntimeError(
        f"Historical readiness checkpoint failed: {readiness.shape}"
    )


readiness_lookup = dict(
    zip(
        readiness[
            "item"
        ],
        readiness[
            "value"
        ],
    )
)


if float(
    readiness_lookup[
        "counts_min_sample"
    ]
) != 1.0:
    raise RuntimeError(
        "Historical counts-min sample checkpoint failed."
    )


if float(
    readiness_lookup[
        "counts_max_sample"
    ]
) != 389.0:
    raise RuntimeError(
        "Historical counts-max sample checkpoint failed."
    )


if not bool(
    readiness_lookup[
        "counts_integer_like_sample"
    ]
):
    raise RuntimeError(
        "Historical integer-like counts checkpoint failed."
    )


display(
    readiness
)

print(
    "PASS: historical readiness checkpoint reproduced without loading the counts layer."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_readiness_HVG8000_r0_6.tsv shape=(15, 2)


,item,value
0,n_cells,351427
1,n_genes,15176
2,input_h5ad_exists,True
3,cluster_key,leiden_scvi_main
4,final_compartment_col,final_compartment_major
5,final_annotation_col,final_annotation_refined
6,condition_col,condition
7,sample_col,sample_id
8,patient_col,patient_id
9,dataset_col,core_dataset


PASS: historical readiness checkpoint reproduced without loading the counts layer.


In [7]:
# =========================
# 6. Cluster-to-state QC summary
# =========================
qc_cols = [
    CLUSTER_KEY,
    "malignant_state",
    FINAL_COMPARTMENT_COL,
    FINAL_ANNOTATION_COL,
]


cluster_state_summary = (
    obs[
        qc_cols
    ]
    .groupby(
        [
            CLUSTER_KEY,
            "malignant_state",
            FINAL_COMPARTMENT_COL,
            FINAL_ANNOTATION_COL,
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_cells"
    )
)


if COPYKAT_COL in obs.columns:
    ck = (
        obs.groupby(
            [
                CLUSTER_KEY,
                COPYKAT_COL,
            ],
            observed=True,
        )
        .size()
        .reset_index(
            name="n"
        )
    )

    ck_tot = (
        ck.groupby(
            CLUSTER_KEY,
            observed=True,
        )[
            "n"
        ]
        .sum()
        .reset_index(
            name="copykat_total"
        )
    )

    ck = ck.merge(
        ck_tot,
        on=CLUSTER_KEY,
        how="left",
    )

    ck[
        "fraction"
    ] = (
        ck[
            "n"
        ]
        / ck[
            "copykat_total"
        ]
    )

    ck_piv = (
        ck.pivot_table(
            index=CLUSTER_KEY,
            columns=COPYKAT_COL,
            values="fraction",
            fill_value=0,
        )
        .reset_index()
    )

    ck_piv.columns = [
        (
            str(
                column
            )
            if column == CLUSTER_KEY
            else f"copykat_fraction_{column}"
        )
        for column in ck_piv.columns
    ]

    cluster_state_summary = (
        cluster_state_summary.merge(
            ck_piv,
            on=CLUSTER_KEY,
            how="left",
        )
    )


CLUSTER_STATE_TSV = (
    METADATA_DIR
    / "malignant_state_cluster_summary_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    cluster_state_summary,
    CLUSTER_STATE_TSV,
)

manifest_records.append(
    {
        "file": str(
            CLUSTER_STATE_TSV
        ),
        "description": "Cluster-to-malignant-state QC summary",
    }
)


if cluster_state_summary.shape != (
    18,
    9,
):
    raise RuntimeError(
        "Historical cluster/state summary checkpoint failed: "
        f"{cluster_state_summary.shape} != (18, 9)"
    )


display(
    cluster_state_summary
)

print(
    "PASS: historical cluster/state QC summary shape matches."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_cluster_summary_HVG8000_r0_6.tsv shape=(18, 9)


,leiden_scvi_main,malignant_state,final_compartment_major,final_annotation_refined,n_cells,copykat_fraction_aneuploid,copykat_fraction_diploid,copykat_fraction_missing,copykat_fraction_not_defined
0,0,OPC_NPC_like_malignant,malignant_like,malignant_OPC_NPC_like_CNV_supported,61646,0.744736,0.145135,0.000000,0.110129
1,1,not_malignant_state,review,review_myeloid_C1Q_TREM2_APOE_TAM_mixed_CNV,59791,0.491746,0.415497,0.000000,0.092756
2,10,not_malignant_state,review,review_myeloid_FCN1_inflammatory_monocyte_CNV_...,6340,0.701104,0.264196,0.000000,0.034700
3,11,NPC_neuronal_like_malignant,malignant_like,malignant_NPC_neuronal_like_CNV_supported,6183,0.589681,0.223840,0.000000,0.186479
4,12,not_malignant_state,review,review_MES_ECM_stromal_like_ambiguous_CNV_high,4803,0.620029,0.301062,0.000000,0.078909
5,13,not_malignant_state,non_malignant_or_diploid_leaning,non_malignant_mural_pericyte_fibroblast_like,3566,0.242569,0.628435,0.000000,0.128996
6,14,not_malignant_state,non_malignant_or_diploid_leaning,non_malignant_endothelial_reference_like,2640,0.168182,0.652273,0.000000,0.179545
7,15,not_malignant_state,non_malignant_or_diploid_leaning,non_malignant_B_plasma_like,1167,0.214225,0.628106,0.000000,0.157669
8,16,not_malignant_state,non_malignant_or_diploid_leaning,non_malignant_rare_mast_cell_like_low_count,183,0.180328,0.688525,0.000000,0.131148
9,17,not_malignant_state,review,review_erythroid_Hb_high_or_ambient_RBC,105,0.400000,0.390476,0.000000,0.209524


PASS: historical cluster/state QC summary shape matches.


In [8]:
# =========================
# 7. Sample-level malignant-state composition
# =========================
mal_cells = (
    obs.loc[
        obs[
            "is_malignant_state_analysis_cell"
        ]
    ]
    .copy()
)


id_cols = [
    sample_col,
    CONDITION_COL,
]

if patient_col:
    id_cols.append(
        patient_col
    )

if dataset_col:
    id_cols.append(
        dataset_col
    )


state_counts = (
    mal_cells.groupby(
        id_cols
        + [
            "malignant_state"
        ],
        observed=True,
    )
    .size()
    .reset_index(
        name="n_state_cells"
    )
)


sample_totals = (
    mal_cells.groupby(
        id_cols,
        observed=True,
    )
    .size()
    .reset_index(
        name="n_malignant_state_analysis_cells"
    )
)


total_sample_cells = (
    obs.groupby(
        id_cols,
        observed=True,
    )
    .size()
    .reset_index(
        name="n_total_cells_in_sample"
    )
)


all_samples = (
    sample_totals[
        id_cols
    ]
    .drop_duplicates()
)


all_states = pd.DataFrame(
    {
        "malignant_state": sorted(
            CLUSTER_TO_MALIGNANT_STATE.values()
        )
    }
)


all_grid = all_samples.merge(
    all_states,
    how="cross",
)


sample_state = all_grid.merge(
    state_counts,
    on=id_cols
    + [
        "malignant_state"
    ],
    how="left",
)


sample_state = sample_state.merge(
    sample_totals,
    on=id_cols,
    how="left",
)


sample_state = sample_state.merge(
    total_sample_cells,
    on=id_cols,
    how="left",
)


sample_state[
    "n_state_cells"
] = (
    sample_state[
        "n_state_cells"
    ]
    .fillna(
        0
    )
    .astype(
        int
    )
)


sample_state[
    "fraction_within_malignant_state_pool"
] = (
    sample_state[
        "n_state_cells"
    ]
    / sample_state[
        "n_malignant_state_analysis_cells"
    ]
)


sample_state[
    "fraction_of_all_sample_cells"
] = (
    sample_state[
        "n_state_cells"
    ]
    / sample_state[
        "n_total_cells_in_sample"
    ]
)


SAMPLE_STATE_TSV = (
    METADATA_DIR
    / "malignant_state_sample_level_proportions_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    sample_state,
    SAMPLE_STATE_TSV,
)

manifest_records.append(
    {
        "file": str(
            SAMPLE_STATE_TSV
        ),
        "description": "Sample-level malignant state proportions",
    }
)


if sample_state.shape != (
    400,
    10,
):
    raise RuntimeError(
        "Historical sample/state composition checkpoint failed: "
        f"{sample_state.shape} != (400, 10)"
    )


if sample_state[
    sample_col
].nunique() != 80:
    raise RuntimeError(
        "Historical malignant-state sample-count checkpoint failed: "
        f"{sample_state[sample_col].nunique()} != 80"
    )


display(
    sample_state.head()
)

print(
    "n samples:",
    sample_state[
        sample_col
    ].nunique(),
)

print(
    "PASS: historical sample-level malignant-state composition reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_sample_level_proportions_HVG8000_r0_6.tsv shape=(400, 10)


,sample_id,condition,patient_id,core_dataset,malignant_state,n_state_cells,n_malignant_state_analysis_cells,n_total_cells_in_sample,fraction_within_malignant_state_pool,fraction_of_all_sample_cells
0,GSM5518596_rGBM-01-A,rGBM,01,DS3_GSE182109,AC_like_reactive_malignant,12,169,4324,0.071006,0.002775
1,GSM5518596_rGBM-01-A,rGBM,01,DS3_GSE182109,NPC_neuronal_like_malignant,0,169,4324,0.000000,0.000000
2,GSM5518596_rGBM-01-A,rGBM,01,DS3_GSE182109,OPC_NPC_like_malignant,45,169,4324,0.266272,0.010407
3,GSM5518596_rGBM-01-A,rGBM,01,DS3_GSE182109,cycling_malignant_candidate,46,169,4324,0.272189,0.010638
4,GSM5518596_rGBM-01-A,rGBM,01,DS3_GSE182109,hypoxia_stress_malignant,66,169,4324,0.390533,0.015264


n samples: 80
PASS: historical sample-level malignant-state composition reproduced.


In [9]:
# =========================
# 8. Condition summaries + non-parametric tests
# =========================
summary_rows = []


for state, sub in sample_state.groupby(
    "malignant_state",
    observed=True,
):
    for condition, group in sub.groupby(
        CONDITION_COL,
        observed=True,
    ):
        vals = (
            group[
                "fraction_within_malignant_state_pool"
            ]
            .dropna()
            .astype(
                float
            )
        )

        summary_rows.append(
            {
                "malignant_state": state,
                "condition": condition,
                "n_samples": int(
                    vals.shape[
                        0
                    ]
                ),
                "mean_fraction": (
                    float(
                        vals.mean()
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "median_fraction": (
                    float(
                        vals.median()
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "q25_fraction": (
                    float(
                        vals.quantile(
                            0.25
                        )
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "q75_fraction": (
                    float(
                        vals.quantile(
                            0.75
                        )
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "min_fraction": (
                    float(
                        vals.min()
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "max_fraction": (
                    float(
                        vals.max()
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
            }
        )


condition_summary = pd.DataFrame(
    summary_rows
)


CONDITION_SUMMARY_TSV = (
    METADATA_DIR
    / "malignant_state_condition_summary_sample_level_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    condition_summary,
    CONDITION_SUMMARY_TSV,
)

manifest_records.append(
    {
        "file": str(
            CONDITION_SUMMARY_TSV
        ),
        "description": "Condition-level summary of malignant-state sample fractions",
    }
)


if condition_summary.shape != (
    20,
    9,
):
    raise RuntimeError(
        "Historical malignant-state condition-summary checkpoint failed: "
        f"{condition_summary.shape} != (20, 9)"
    )


expected_condition_samples = {
    "GBM": 39,
    "LGG": 4,
    "ndGBM": 20,
    "rGBM": 17,
}


for condition, expected in expected_condition_samples.items():
    observed = set(
        condition_summary.loc[
            condition_summary[
                "condition"
            ]
            .astype(str)
            .eq(
                condition
            ),
            "n_samples",
        ]
        .astype(int)
        .tolist()
    )

    if observed != {
        expected
    }:
        raise RuntimeError(
            f"Historical state-analysis sample checkpoint failed for {condition}: "
            f"{observed} != {{{expected}}}"
        )


test_col = (
    "fraction_within_malignant_state_pool"
)


kw_rows = []


for state, sub in sample_state.groupby(
    "malignant_state",
    observed=True,
):
    groups = []
    group_names = []

    for condition, group in sub.groupby(
        CONDITION_COL,
        observed=True,
    ):
        vals = (
            group[
                test_col
            ]
            .dropna()
            .astype(
                float
            )
        )

        if len(
            vals
        ) > 0:
            groups.append(
                vals
            )

            group_names.append(
                condition
            )

    p_value = safe_kruskal(
        groups
    )

    kw_rows.append(
        {
            "malignant_state": state,
            "test": "Kruskal-Wallis",
            "value_tested": test_col,
            "conditions_tested": ",".join(
                map(
                    str,
                    group_names,
                )
            ),
            "n_conditions": len(
                group_names
            ),
            "p_value": p_value,
        }
    )


kw_df = pd.DataFrame(
    kw_rows
)

kw_df[
    "q_value_bh"
] = bh_fdr(
    kw_df[
        "p_value"
    ]
)


KW_TSV = (
    METADATA_DIR
    / "malignant_state_global_kruskal_sample_level_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    kw_df,
    KW_TSV,
)

manifest_records.append(
    {
        "file": str(
            KW_TSV
        ),
        "description": "Kruskal-Wallis tests for malignant-state proportions by condition",
    }
)


if kw_df.shape != (
    5,
    7,
):
    raise RuntimeError(
        f"Historical Kruskal-Wallis checkpoint failed: {kw_df.shape}"
    )


pair_rows = []


conditions = sorted(
    sample_state[
        CONDITION_COL
    ]
    .dropna()
    .astype(str)
    .unique()
)


for state, sub in sample_state.groupby(
    "malignant_state",
    observed=True,
):
    sub = sub.copy()

    sub[
        CONDITION_COL
    ] = sub[
        CONDITION_COL
    ].astype(str)

    for i in range(
        len(
            conditions
        )
    ):
        for j in range(
            i + 1,
            len(
                conditions
            ),
        ):
            c1 = conditions[
                i
            ]

            c2 = conditions[
                j
            ]

            x = sub.loc[
                sub[
                    CONDITION_COL
                ]
                == c1,
                test_col,
            ].astype(
                float
            )

            y = sub.loc[
                sub[
                    CONDITION_COL
                ]
                == c2,
                test_col,
            ].astype(
                float
            )

            p_value = safe_mannwhitney(
                x,
                y,
            )

            pair_rows.append(
                {
                    "malignant_state": state,
                    "test": "Mann-Whitney U",
                    "value_tested": test_col,
                    "condition_1": c1,
                    "condition_2": c2,
                    "n_samples_1": int(
                        x.dropna().shape[
                            0
                        ]
                    ),
                    "n_samples_2": int(
                        y.dropna().shape[
                            0
                        ]
                    ),
                    "median_1": (
                        float(
                            x.median()
                        )
                        if len(
                            x.dropna()
                        )
                        else np.nan
                    ),
                    "median_2": (
                        float(
                            y.median()
                        )
                        if len(
                            y.dropna()
                        )
                        else np.nan
                    ),
                    "delta_median_1_minus_2": (
                        float(
                            x.median()
                            - y.median()
                        )
                        if (
                            len(
                                x.dropna()
                            )
                            and len(
                                y.dropna()
                            )
                        )
                        else np.nan
                    ),
                    "p_value": p_value,
                }
            )


pair_df = pd.DataFrame(
    pair_rows
)

pair_df[
    "q_value_bh"
] = bh_fdr(
    pair_df[
        "p_value"
    ]
)


PAIR_TSV = (
    METADATA_DIR
    / "malignant_state_pairwise_mannwhitney_sample_level_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    pair_df,
    PAIR_TSV,
)

manifest_records.append(
    {
        "file": str(
            PAIR_TSV
        ),
        "description": "Pairwise Mann-Whitney tests for malignant-state proportions by condition",
    }
)


if pair_df.shape != (
    30,
    12,
):
    raise RuntimeError(
        f"Historical pairwise-test checkpoint failed: {pair_df.shape}"
    )


display(
    condition_summary
)

display(
    kw_df.sort_values(
        "q_value_bh"
    )
)

display(
    pair_df.sort_values(
        "q_value_bh"
    ).head(
        30
    )
)


print(
    "PASS: historical condition summaries and non-parametric tests reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_condition_summary_sample_level_HVG8000_r0_6.tsv shape=(20, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_global_kruskal_sample_level_HVG8000_r0_6.tsv shape=(5, 7)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_pairwise_mannwhitney_sample_level_HVG8000_r0_6.tsv shape=(30, 12)


,malignant_state,condition,n_samples,mean_fraction,median_fraction,q25_fraction,q75_fraction,min_fraction,max_fraction
0,AC_like_reactive_malignant,GBM,39,0.172910,0.093506,0.054287,0.283868,0.014368,0.886364
1,AC_like_reactive_malignant,LGG,4,0.081874,0.026239,0.021165,0.086948,0.016039,0.258976
2,AC_like_reactive_malignant,ndGBM,20,0.256650,0.151198,0.092224,0.438646,0.009372,0.971963
3,AC_like_reactive_malignant,rGBM,17,0.115190,0.094192,0.085191,0.135269,0.054148,0.222156
4,NPC_neuronal_like_malignant,GBM,39,0.065669,0.017964,0.003866,0.061262,0.000000,0.445688
5,NPC_neuronal_like_malignant,LGG,4,0.060776,0.063397,0.019264,0.104909,0.001177,0.115132
6,NPC_neuronal_like_malignant,ndGBM,20,0.032690,0.005670,0.000563,0.013916,0.000000,0.398316
7,NPC_neuronal_like_malignant,rGBM,17,0.013517,0.004968,0.001166,0.011545,0.000000,0.082096
8,OPC_NPC_like_malignant,GBM,39,0.324888,0.299812,0.203177,0.436142,0.000000,0.734240
9,OPC_NPC_like_malignant,LGG,4,0.786485,0.781077,0.729883,0.837679,0.649794,0.933991


,malignant_state,test,value_tested,conditions_tested,n_conditions,p_value,q_value_bh
3,cycling_malignant_candidate,Kruskal-Wallis,fraction_within_malignant_state_pool,"GBM,LGG,ndGBM,rGBM",4,0.008326,0.020816
2,OPC_NPC_like_malignant,Kruskal-Wallis,fraction_within_malignant_state_pool,"GBM,LGG,ndGBM,rGBM",4,0.005453,0.020816
4,hypoxia_stress_malignant,Kruskal-Wallis,fraction_within_malignant_state_pool,"GBM,LGG,ndGBM,rGBM",4,0.015338,0.025564
1,NPC_neuronal_like_malignant,Kruskal-Wallis,fraction_within_malignant_state_pool,"GBM,LGG,ndGBM,rGBM",4,0.054499,0.068123
0,AC_like_reactive_malignant,Kruskal-Wallis,fraction_within_malignant_state_pool,"GBM,LGG,ndGBM,rGBM",4,0.082393,0.082393


,malignant_state,test,value_tested,condition_1,condition_2,n_samples_1,n_samples_2,median_1,median_2,delta_median_1_minus_2,p_value,q_value_bh
12,OPC_NPC_like_malignant,Mann-Whitney U,fraction_within_malignant_state_pool,GBM,LGG,39,4,0.299812,0.781077,-0.481265,0.000194,0.005834
24,hypoxia_stress_malignant,Mann-Whitney U,fraction_within_malignant_state_pool,GBM,LGG,39,4,0.127023,0.003070,0.123953,0.000616,0.009238
16,OPC_NPC_like_malignant,Mann-Whitney U,fraction_within_malignant_state_pool,LGG,rGBM,4,17,0.781077,0.388652,0.392425,0.001337,0.010025
28,hypoxia_stress_malignant,Mann-Whitney U,fraction_within_malignant_state_pool,LGG,rGBM,4,17,0.003070,0.184945,-0.181875,0.001337,0.010025
15,OPC_NPC_like_malignant,Mann-Whitney U,fraction_within_malignant_state_pool,LGG,ndGBM,4,20,0.781077,0.406923,0.374154,0.002211,0.013266
22,cycling_malignant_candidate,Mann-Whitney U,fraction_within_malignant_state_pool,LGG,rGBM,4,17,0.069373,0.292683,-0.223310,0.004010,0.020050
18,cycling_malignant_candidate,Mann-Whitney U,fraction_within_malignant_state_pool,GBM,LGG,39,4,0.203187,0.069373,0.133815,0.018199,0.077998
8,NPC_neuronal_like_malignant,Mann-Whitney U,fraction_within_malignant_state_pool,GBM,rGBM,39,17,0.017964,0.004968,0.012997,0.030726,0.092179
20,cycling_malignant_candidate,Mann-Whitney U,fraction_within_malignant_state_pool,GBM,rGBM,39,17,0.203187,0.292683,-0.089496,0.027131,0.092179
23,cycling_malignant_candidate,Mann-Whitney U,fraction_within_malignant_state_pool,ndGBM,rGBM,20,17,0.196490,0.292683,-0.096193,0.029137,0.092179


PASS: historical condition summaries and non-parametric tests reproduced.


In [10]:
# =========================
# 9. Program marker presence
# =========================
marker_rows = []


for program, genes in PROGRAM_MARKERS.items():
    present = [
        gene
        for gene in genes
        if gene in var_name_set
    ]

    missing = [
        gene
        for gene in genes
        if gene not in var_name_set
    ]

    marker_rows.append(
        {
            "program": program,
            "n_genes_requested": len(
                genes
            ),
            "n_genes_present": len(
                present
            ),
            "n_genes_missing": len(
                missing
            ),
            "present_genes": ",".join(
                present
            ),
            "missing_genes": ",".join(
                missing
            ),
        }
    )


marker_presence = pd.DataFrame(
    marker_rows
)


MARKER_PRESENCE_TSV = (
    METADATA_DIR
    / "malignant_state_program_marker_presence_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    marker_presence,
    MARKER_PRESENCE_TSV,
)

manifest_records.append(
    {
        "file": str(
            MARKER_PRESENCE_TSV
        ),
        "description": "Presence/absence of malignant program marker genes",
    }
)


if marker_presence.shape != (
    7,
    6,
):
    raise RuntimeError(
        f"Historical marker-presence checkpoint failed: {marker_presence.shape}"
    )


if not (
    marker_presence[
        "n_genes_missing"
    ]
    .astype(int)
    .eq(
        0
    )
    .all()
):
    raise RuntimeError(
        "Historical program marker panel is no longer complete."
    )


display(
    marker_presence
)

print(
    "PASS: all historical program-marker genes are present."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_program_marker_presence_HVG8000_r0_6.tsv shape=(7, 6)


,program,n_genes_requested,n_genes_present,n_genes_missing,present_genes,missing_genes
0,OPC_proneural_program,9,9,0,"PDGFRA,OLIG1,OLIG2,PTPRZ1,BCAN,SOX6,SOX10,CSPG...",
1,NPC_neural_program,9,9,0,"SOX4,SOX11,DCX,STMN2,TUBB3,DLX5,MYT1L,ASCL1,NE...",
2,AC_like_reactive_program,8,8,0,"GFAP,AQP4,ALDH1L1,SLC1A3,CLU,SPARCL1,APOE,VIM",
3,MES_like_program,9,9,0,"CHI3L1,CD44,VIM,FN1,SERPINE1,TGFBI,LGALS3,CCL2...",
4,hypoxia_stress_program,9,9,0,"CA9,VEGFA,HILPDA,LOX,BNIP3,PDK1,SLC2A1,DDIT4,N...",
5,cycling_program,9,9,0,"MKI67,TOP2A,PBK,BIRC5,UBE2C,CENPF,NUSAP1,CENPA...",
6,myeloid_contamination_review_program,9,9,0,"C1QA,C1QB,C1QC,TYROBP,LYZ,FCN1,CD74,HLA-DRA,CSF1R",


PASS: all historical program-marker genes are present.


In [11]:
# =========================
# 10. Recover or stream malignant-cell raw counts scratch H5AD
# =========================
mal_mask = (
    obs[
        "is_malignant_state_analysis_cell"
    ]
    .to_numpy(
        dtype=bool
    )
)

n_malignant = int(
    mal_mask.sum()
)

if n_malignant != 161884:
    raise RuntimeError(
        "Historical malignant-state analysis cell-count checkpoint failed: "
        f"{n_malignant} != 161884"
    )


row_nnz = np.diff(
    full_indptr
)

selected_nnz = int(
    row_nnz[
        mal_mask
    ].sum()
)


# Keep the scoring scratch metadata intentionally minimal.
scoring_obs_cols = [
    "malignant_state",
    CONDITION_COL,
    CLUSTER_KEY,
    sample_col,
]

if (
    patient_col
    and patient_col not in scoring_obs_cols
):
    scoring_obs_cols.append(
        patient_col
    )

if (
    dataset_col
    and dataset_col not in scoring_obs_cols
):
    scoring_obs_cols.append(
        dataset_col
    )


scoring_obs = (
    obs.loc[
        mal_mask,
        scoring_obs_cols,
    ]
    .copy()
)

scoring_var = pd.DataFrame(
    index=var.index.copy()
)


estimated_raw_bytes = (
    selected_nnz
    * (
        np.dtype(
            counts_data_dtype
        ).itemsize
        + np.dtype(
            counts_indices_dtype
        ).itemsize
    )
    + (
        n_malignant
        + 1
    )
    * np.dtype(
        np.int64
    ).itemsize
)


free_scratch = shutil.disk_usage(
    SCRATCH_DIR
).free


print(
    "Malignant cells:",
    n_malignant,
)

print(
    "Selected counts nnz:",
    selected_nnz,
)

print(
    "Estimated raw scratch CSR GiB:",
    round(
        estimated_raw_bytes
        / 1024**3,
        3,
    ),
)

print(
    "Scratch free GiB:",
    round(
        free_scratch
        / 1024**3,
        3,
    ),
)


raw_audit = audit_scoring_scratch(
    SCORING_COUNTS_H5AD,
    expected_obs_names=scoring_obs.index,
    expected_n_vars=source_shape[
        1
    ],
    expected_nnz=selected_nnz,
)


if raw_audit[
    "valid"
]:
    print(
        "[RECOVERY PASS] Reusing complete malignant-cell raw-count scratch:"
    )

    print(
        SCORING_COUNTS_H5AD
    )

    print(
        raw_audit
    )

    # Reduce metadata memory if this file came from v1/v2 and retained full obs.
    replace_h5ad_dataframe_group(
        SCORING_COUNTS_H5AD,
        "obs",
        scoring_obs,
    )

    replace_h5ad_dataframe_group(
        SCORING_COUNTS_H5AD,
        "var",
        scoring_var,
    )

else:
    print(
        "[RAW SCRATCH REBUILD]",
        raw_audit,
    )

    if SCORING_COUNTS_H5AD.exists():
        SCORING_COUNTS_H5AD.unlink()


    if free_scratch < int(
        estimated_raw_bytes
        * 1.20
        + 512 * 1024**2
    ):
        raise RuntimeError(
            "Insufficient scratch space for malignant-cell counts-only H5AD."
        )


    shell = ad.AnnData(
        X=sp.csr_matrix(
            (
                n_malignant,
                source_shape[
                    1
                ],
            ),
            dtype=counts_data_dtype,
        ),
        obs=scoring_obs,
        var=scoring_var,
    )

    shell.write_h5ad(
        SCORING_COUNTS_H5AD
    )

    del shell
    gc.collect()


    BLOCK_ROWS = 512


    with h5py.File(
        INPUT_H5AD,
        "r",
    ) as src, h5py.File(
        SCORING_COUNTS_H5AD,
        "r+",
    ) as dst:

        src_counts = src[
            "layers"
        ][
            counts_layer
        ]

        if "X" in dst:
            del dst[
                "X"
            ]

        x_group = dst.create_group(
            "X"
        )

        for key, value in src_counts.attrs.items():
            x_group.attrs[
                key
            ] = value

        x_group.attrs[
            "encoding-type"
        ] = "csr_matrix"

        x_group.attrs[
            "encoding-version"
        ] = "0.1.0"

        x_group.attrs[
            "shape"
        ] = np.asarray(
            [
                n_malignant,
                source_shape[
                    1
                ],
            ],
            dtype=np.int64,
        )


        chunk_values = min(
            max(
                100_000,
                selected_nnz
                // 256,
            ),
            2_000_000,
        )


        out_data = x_group.create_dataset(
            "data",
            shape=(
                selected_nnz,
            ),
            dtype=counts_data_dtype,
            chunks=(
                min(
                    chunk_values,
                    max(
                        selected_nnz,
                        1,
                    ),
                ),
            ),
        )


        out_indices = x_group.create_dataset(
            "indices",
            shape=(
                selected_nnz,
            ),
            dtype=counts_indices_dtype,
            chunks=(
                min(
                    chunk_values,
                    max(
                        selected_nnz,
                        1,
                    ),
                ),
            ),
        )


        out_indptr = np.empty(
            n_malignant
            + 1,
            dtype=np.int64,
        )

        out_indptr[
            0
        ] = 0

        out_row_cursor = 0
        out_nnz_cursor = 0
        next_report = 5


        for row0 in range(
            0,
            source_shape[
                0
            ],
            BLOCK_ROWS,
        ):
            row1 = min(
                row0
                + BLOCK_ROWS,
                source_shape[
                    0
                ],
            )

            block_mask = mal_mask[
                row0:row1
            ]

            if not block_mask.any():
                continue

            ptr0 = int(
                full_indptr[
                    row0
                ]
            )

            ptr1 = int(
                full_indptr[
                    row1
                ]
            )

            block_data = np.asarray(
                src_counts[
                    "data"
                ][
                    ptr0:ptr1
                ]
            )

            block_indices = np.asarray(
                src_counts[
                    "indices"
                ][
                    ptr0:ptr1
                ]
            )

            block_indptr = (
                full_indptr[
                    row0:row1
                    + 1
                ]
                - ptr0
            )

            block_csr = sp.csr_matrix(
                (
                    block_data,
                    block_indices,
                    block_indptr,
                ),
                shape=(
                    row1
                    - row0,
                    source_shape[
                        1
                    ],
                ),
            )

            selected = block_csr[
                block_mask
            ]

            block_nnz = int(
                selected.nnz
            )

            block_rows_selected = int(
                selected.shape[
                    0
                ]
            )

            if block_nnz:
                out_data[
                    out_nnz_cursor:
                    out_nnz_cursor
                    + block_nnz
                ] = selected.data

                out_indices[
                    out_nnz_cursor:
                    out_nnz_cursor
                    + block_nnz
                ] = selected.indices

            local_indptr = (
                selected.indptr[
                    1:
                ]
                .astype(
                    np.int64,
                    copy=False,
                )
                + out_nnz_cursor
            )

            out_indptr[
                out_row_cursor
                + 1:
                out_row_cursor
                + block_rows_selected
                + 1
            ] = local_indptr

            out_nnz_cursor += block_nnz
            out_row_cursor += block_rows_selected

            pct = int(
                row1
                * 100
                / source_shape[
                    0
                ]
            )

            if pct >= next_report:
                print(
                    f"  streamed source rows: {pct}% | "
                    f"selected rows: {out_row_cursor:,} | "
                    f"nnz: {out_nnz_cursor:,}"
                )

                next_report += 5


        if out_row_cursor != n_malignant:
            raise RuntimeError(
                "Scratch CSR row-count checkpoint failed: "
                f"{out_row_cursor} != {n_malignant}"
            )


        if out_nnz_cursor != selected_nnz:
            raise RuntimeError(
                "Scratch CSR nnz checkpoint failed: "
                f"{out_nnz_cursor} != {selected_nnz}"
            )


        x_group.create_dataset(
            "indptr",
            data=out_indptr,
        )

        dst.flush()


    print(
        "[SAVED COUNTS-ONLY SCORING H5AD]",
        SCORING_COUNTS_H5AD,
    )


raw_audit = audit_scoring_scratch(
    SCORING_COUNTS_H5AD,
    expected_obs_names=scoring_obs.index,
    expected_n_vars=source_shape[
        1
    ],
    expected_nnz=selected_nnz,
)

if not raw_audit[
    "valid"
]:
    raise RuntimeError(
        "Raw malignant-cell scratch validation failed after preparation: "
        f"{raw_audit}"
    )


print(
    "PASS: exact malignant-cell raw-count scratch is available."
)


Malignant cells: 161884
Selected counts nnz: 383992077
Estimated raw scratch CSR GiB: 2.862
Scratch free GiB: 27.585
[RECOVERY PASS] Reusing complete malignant-cell raw-count scratch:
E:\glioma-cnv-scratch\main_analysis_20\20_malignant_cells_fullgene_counts_only_for_scoring.h5ad
{'valid': True, 'reason': 'ok', 'data_dtype': 'int32', 'normalized_rows_complete': 0, 'normalization_complete': False}
PASS: exact malignant-cell raw-count scratch is available.


In [12]:
# =========================
# 11. Resumable blockwise Scanpy normalization/log1p + historical score_genes
# =========================
NORMALIZE_BLOCK_ROWS = 512


def initialize_normalized_scratch():
    if SCORING_NORM_H5AD.exists():
        SCORING_NORM_H5AD.unlink()

    shell = ad.AnnData(
        X=sp.csr_matrix(
            (
                n_malignant,
                source_shape[
                    1
                ],
            ),
            dtype=np.float32,
        ),
        obs=scoring_obs,
        var=scoring_var,
    )

    shell.write_h5ad(
        SCORING_NORM_H5AD
    )

    del shell
    gc.collect()


    with h5py.File(
        SCORING_COUNTS_H5AD,
        "r",
    ) as src, h5py.File(
        SCORING_NORM_H5AD,
        "r+",
    ) as dst:

        raw_x = src[
            "X"
        ]

        raw_indptr = np.asarray(
            raw_x[
                "indptr"
            ][:]
        )

        if "X" in dst:
            del dst[
                "X"
            ]

        x_group = dst.create_group(
            "X"
        )

        x_group.attrs[
            "encoding-type"
        ] = "csr_matrix"

        x_group.attrs[
            "encoding-version"
        ] = "0.1.0"

        x_group.attrs[
            "shape"
        ] = np.asarray(
            [
                n_malignant,
                source_shape[
                    1
                ],
            ],
            dtype=np.int64,
        )

        x_group.attrs[
            "normalized_rows_complete"
        ] = 0

        x_group.attrs[
            "normalization_complete"
        ] = False

        x_group.attrs[
            "normalization_target_sum"
        ] = 10000.0

        x_group.attrs[
            "normalization_method"
        ] = "scanpy_normalize_total_then_log1p_blockwise"


        chunk_values = min(
            max(
                100_000,
                selected_nnz
                // 256,
            ),
            2_000_000,
        )


        x_group.create_dataset(
            "data",
            shape=(
                selected_nnz,
            ),
            dtype=np.float32,
            chunks=(
                min(
                    chunk_values,
                    max(
                        selected_nnz,
                        1,
                    ),
                ),
            ),
        )


        x_group.create_dataset(
            "indices",
            shape=(
                selected_nnz,
            ),
            dtype=raw_x[
                "indices"
            ].dtype,
            chunks=(
                min(
                    chunk_values,
                    max(
                        selected_nnz,
                        1,
                    ),
                ),
            ),
        )


        x_group.create_dataset(
            "indptr",
            data=raw_indptr.astype(
                np.int64,
                copy=False,
            ),
        )

        dst.flush()


norm_exists = SCORING_NORM_H5AD.exists()

if not norm_exists:
    print(
        "[NORMALIZED SCRATCH INIT]"
    )

    initialize_normalized_scratch()


# Inspect normalized scratch. A partially normalized file is resumable.
norm_audit = audit_scoring_scratch(
    SCORING_NORM_H5AD,
    expected_obs_names=scoring_obs.index,
    expected_n_vars=source_shape[
        1
    ],
    expected_nnz=selected_nnz,
    require_float32=True,
    require_complete_normalization=False,
)


if not norm_audit[
    "valid"
]:
    print(
        "[NORMALIZED SCRATCH INVALID — RECREATE]",
        norm_audit,
    )

    initialize_normalized_scratch()

    norm_audit = audit_scoring_scratch(
        SCORING_NORM_H5AD,
        expected_obs_names=scoring_obs.index,
        expected_n_vars=source_shape[
            1
        ],
        expected_nnz=selected_nnz,
        require_float32=True,
        require_complete_normalization=False,
    )

    if not norm_audit[
        "valid"
    ]:
        raise RuntimeError(
            "Could not initialize a valid normalized scratch H5AD."
        )


with h5py.File(
    SCORING_NORM_H5AD,
    "r",
) as handle:
    normalized_rows_complete = int(
        handle[
            "X"
        ].attrs.get(
            "normalized_rows_complete",
            0,
        )
    )

    normalization_complete = bool(
        handle[
            "X"
        ].attrs.get(
            "normalization_complete",
            False,
        )
    )


if (
    normalization_complete
    and normalized_rows_complete
    == n_malignant
):
    print(
        "[RECOVERY PASS] Reusing complete normalized/log1p scratch:"
    )

    print(
        SCORING_NORM_H5AD
    )

else:
    print(
        "[NORMALIZATION RESUME] completed rows:",
        normalized_rows_complete,
        "/",
        n_malignant,
    )


    # Resume only at a completed block boundary. The progress attribute is
    # updated and flushed only after each block has been fully written.
    start_row = normalized_rows_complete


    with h5py.File(
        SCORING_COUNTS_H5AD,
        "r",
    ) as src, h5py.File(
        SCORING_NORM_H5AD,
        "r+",
    ) as dst:

        raw_x = src[
            "X"
        ]

        norm_x = dst[
            "X"
        ]

        raw_indptr = np.asarray(
            raw_x[
                "indptr"
            ][:]
        )

        next_report = max(
            5,
            (
                start_row
                * 100
                // n_malignant
                // 5
                + 1
            )
            * 5,
        )


        for row0 in range(
            start_row,
            n_malignant,
            NORMALIZE_BLOCK_ROWS,
        ):
            row1 = min(
                row0
                + NORMALIZE_BLOCK_ROWS,
                n_malignant,
            )

            ptr0 = int(
                raw_indptr[
                    row0
                ]
            )

            ptr1 = int(
                raw_indptr[
                    row1
                ]
            )

            block_data = np.asarray(
                raw_x[
                    "data"
                ][
                    ptr0:ptr1
                ]
            )

            block_indices = np.asarray(
                raw_x[
                    "indices"
                ][
                    ptr0:ptr1
                ]
            )

            block_indptr = (
                raw_indptr[
                    row0:row1
                    + 1
                ]
                - ptr0
            )


            block_csr = sp.csr_matrix(
                (
                    block_data,
                    block_indices,
                    block_indptr,
                ),
                shape=(
                    row1
                    - row0,
                    source_shape[
                        1
                    ],
                ),
            )


            block_adata = ad.AnnData(
                X=block_csr
            )


            # Exact historical Scanpy calls, applied to independent row blocks.
            # Both operations are strictly cell/element-wise, so block boundaries
            # do not alter their mathematical result.
            sc.pp.normalize_total(
                block_adata,
                target_sum=1e4,
            )

            sc.pp.log1p(
                block_adata
            )


            block_norm = block_adata.X.tocsr()


            if block_norm.nnz != (
                ptr1
                - ptr0
            ):
                raise RuntimeError(
                    "Blockwise normalization unexpectedly changed CSR sparsity."
                )


            if not np.array_equal(
                block_norm.indices,
                block_indices,
            ):
                raise RuntimeError(
                    "Blockwise normalization unexpectedly changed CSR index order."
                )


            norm_x[
                "data"
            ][
                ptr0:ptr1
            ] = block_norm.data.astype(
                np.float32,
                copy=False,
            )

            norm_x[
                "indices"
            ][
                ptr0:ptr1
            ] = block_indices


            # Commit progress only after the whole block has been written.
            norm_x.attrs[
                "normalized_rows_complete"
            ] = int(
                row1
            )

            dst.flush()


            del block_adata
            del block_norm
            del block_csr
            del block_data
            del block_indices
            gc.collect()


            pct = int(
                row1
                * 100
                / n_malignant
            )

            if pct >= next_report:
                print(
                    f"  normalized/log1p rows: {pct}% "
                    f"({row1:,}/{n_malignant:,})"
                )

                next_report += 5


        norm_x.attrs[
            "normalization_complete"
        ] = True

        norm_x.attrs[
            "normalized_rows_complete"
        ] = int(
            n_malignant
        )

        dst.flush()


norm_audit = audit_scoring_scratch(
    SCORING_NORM_H5AD,
    expected_obs_names=scoring_obs.index,
    expected_n_vars=source_shape[
        1
    ],
    expected_nnz=selected_nnz,
    require_float32=True,
    require_complete_normalization=True,
)


if not norm_audit[
    "valid"
]:
    raise RuntimeError(
        "Normalized/log1p scratch validation failed: "
        f"{norm_audit}"
    )


print(
    "PASS: complete normalized/log1p scratch is available."
)


# Raw counts scratch is no longer needed once normalization is complete.
# Removing it now releases local NVMe space while preserving the resumable
# normalized scratch for any later interruption during score_genes/figures.
if SCORING_COUNTS_H5AD.exists():
    try:
        SCORING_COUNTS_H5AD.unlink()

        print(
            "[REMOVED RAW COUNTS SCRATCH]",
            SCORING_COUNTS_H5AD,
        )

    except Exception as error:
        print(
            "[WARNING] Could not remove raw counts scratch:",
            repr(
                error
            ),
        )


gc.collect()


print(
    "[LOAD PRE-NORMALIZED MALIGNANT SCORING OBJECT]"
)

adata_score = sc.read_h5ad(
    SCORING_NORM_H5AD
)


print(
    adata_score
)


if adata_score.shape != (
    161884,
    15176,
):
    raise RuntimeError(
        f"Scoring object shape mismatch: {adata_score.shape}"
    )


if adata_score.X.dtype != np.float32:
    raise RuntimeError(
        f"Normalized scoring matrix dtype mismatch: {adata_score.X.dtype}"
    )


score_cols = []


for program, genes in PROGRAM_MARKERS.items():
    present = [
        gene
        for gene in genes
        if gene in adata_score.var_names
    ]

    score_name = (
        f"score_{program}"
    )

    if len(
        present
    ) >= 2:
        sc.tl.score_genes(
            adata_score,
            gene_list=present,
            score_name=score_name,
            random_state=RANDOM_STATE,
            use_raw=False,
        )

        print(
            "[SCORED]",
            score_name,
            "n_genes=",
            len(
                present
            ),
        )

    else:
        adata_score.obs[
            score_name
        ] = np.nan

        print(
            "[SKIPPED]",
            score_name,
            "not enough genes present",
        )

    score_cols.append(
        score_name
    )

    gc.collect()


if not np.array_equal(
    adata_score.obs_names.astype(str),
    scoring_obs.index.astype(str),
):
    raise RuntimeError(
        "Scoring object cell order does not match canonical malignant-cell order."
    )


for column in score_cols:
    obs[
        column
    ] = np.nan

    obs.loc[
        adata_score.obs_names,
        column,
    ] = (
        adata_score.obs[
            column
        ]
        .astype(
            float
        )
        .to_numpy()
    )


print(
    "score_cols:",
    score_cols,
)


# Historical aggregate score sentinel.
ac_mask = (
    adata_score.obs[
        "malignant_state"
    ]
    .astype(str)
    .eq(
        "AC_like_reactive_malignant"
    )
)


historical_ac_score_sentinels = {
    "score_OPC_proneural_program": 0.152990,
    "score_NPC_neural_program": -0.227970,
    "score_AC_like_reactive_program": 1.815580,
    "score_MES_like_program": 0.473419,
    "score_hypoxia_stress_program": -0.052144,
    "score_cycling_program": -0.231907,
    "score_myeloid_contamination_review_program": 0.016158,
}


for column, expected in historical_ac_score_sentinels.items():
    observed = float(
        adata_score.obs.loc[
            ac_mask,
            column,
        ].mean()
    )

    if not np.isclose(
        observed,
        expected,
        atol=5e-6,
        rtol=0,
    ):
        raise RuntimeError(
            "Historical Scanpy score sentinel failed for "
            f"{column}: {observed} != {expected}"
        )


print(
    "PASS: historical Scanpy program-score sentinels reproduced."
)


[NORMALIZED SCRATCH INIT]
[NORMALIZATION RESUME] completed rows: 0 / 161884
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
  normalized/log1p rows: 5% (8,192/161,884)
normalizing counts per cell
    finished (0:00:00)
normalizing c

In [13]:
# =========================
# 12. Program score summaries
# =========================
score_df = (
    obs.loc[
        obs[
            "is_malignant_state_analysis_cell"
        ],
        [
            "malignant_state",
            CONDITION_COL,
            CLUSTER_KEY,
            sample_col,
        ]
        + score_cols,
    ]
    .copy()
)


score_summary_rows = []


for state, sub_state in score_df.groupby(
    "malignant_state",
    observed=True,
):
    for score_col in score_cols:
        vals = (
            sub_state[
                score_col
            ]
            .dropna()
            .astype(
                float
            )
        )

        score_summary_rows.append(
            {
                "level": "malignant_state",
                "malignant_state": state,
                "condition": "all",
                "score": score_col,
                "n_cells": int(
                    vals.shape[
                        0
                    ]
                ),
                "mean_score": (
                    float(
                        vals.mean()
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "median_score": (
                    float(
                        vals.median()
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "q25_score": (
                    float(
                        vals.quantile(
                            0.25
                        )
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
                "q75_score": (
                    float(
                        vals.quantile(
                            0.75
                        )
                    )
                    if len(
                        vals
                    )
                    else np.nan
                ),
            }
        )

    for condition, sub_cond in sub_state.groupby(
        CONDITION_COL,
        observed=True,
    ):
        for score_col in score_cols:
            vals = (
                sub_cond[
                    score_col
                ]
                .dropna()
                .astype(
                    float
                )
            )

            score_summary_rows.append(
                {
                    "level": "malignant_state_by_condition",
                    "malignant_state": state,
                    "condition": condition,
                    "score": score_col,
                    "n_cells": int(
                        vals.shape[
                            0
                        ]
                    ),
                    "mean_score": (
                        float(
                            vals.mean()
                        )
                        if len(
                            vals
                        )
                        else np.nan
                    ),
                    "median_score": (
                        float(
                            vals.median()
                        )
                        if len(
                            vals
                        )
                        else np.nan
                    ),
                    "q25_score": (
                        float(
                            vals.quantile(
                                0.25
                            )
                        )
                        if len(
                            vals
                        )
                        else np.nan
                    ),
                    "q75_score": (
                        float(
                            vals.quantile(
                                0.75
                            )
                        )
                        if len(
                            vals
                        )
                        else np.nan
                    ),
                }
            )


score_summary = pd.DataFrame(
    score_summary_rows
)


SCORE_SUMMARY_TSV = (
    METADATA_DIR
    / "malignant_state_program_score_summary_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    score_summary,
    SCORE_SUMMARY_TSV,
)

manifest_records.append(
    {
        "file": str(
            SCORE_SUMMARY_TSV
        ),
        "description": "Program score summaries by malignant state and condition",
    }
)


if score_summary.shape != (
    175,
    9,
):
    raise RuntimeError(
        f"Historical program-score summary checkpoint failed: {score_summary.shape}"
    )


display(
    score_summary.head(
        30
    )
)

print(
    "PASS: historical program-score summary shape matches."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_program_score_summary_HVG8000_r0_6.tsv shape=(175, 9)


,level,malignant_state,condition,score,n_cells,mean_score,median_score,q25_score,q75_score
0,malignant_state,AC_like_reactive_malignant,all,score_OPC_proneural_program,22525,0.152990,0.153645,-0.069946,0.360296
1,malignant_state,AC_like_reactive_malignant,all,score_NPC_neural_program,22525,-0.227970,-0.255986,-0.354625,-0.130670
2,malignant_state,AC_like_reactive_malignant,all,score_AC_like_reactive_program,22525,1.815580,1.814803,1.427303,2.208908
3,malignant_state,AC_like_reactive_malignant,all,score_MES_like_program,22525,0.473419,0.392979,0.112906,0.790734
4,malignant_state,AC_like_reactive_malignant,all,score_hypoxia_stress_program,22525,-0.052144,-0.104406,-0.204182,0.057991
5,malignant_state,AC_like_reactive_malignant,all,score_cycling_program,22525,-0.231907,-0.251184,-0.335636,-0.158578
6,malignant_state,AC_like_reactive_malignant,all,score_myeloid_contamination_review_program,22525,0.016158,-0.083377,-0.174541,0.132758
7,malignant_state_by_condition,AC_like_reactive_malignant,GBM,score_OPC_proneural_program,13834,0.113276,0.105919,-0.113165,0.321693
8,malignant_state_by_condition,AC_like_reactive_malignant,GBM,score_NPC_neural_program,13834,-0.200155,-0.232517,-0.323923,-0.105482
9,malignant_state_by_condition,AC_like_reactive_malignant,GBM,score_AC_like_reactive_program,13834,1.827467,1.826188,1.417410,2.244132


PASS: historical program-score summary shape matches.


In [14]:
# =========================
# 13. Sample-level program scores
# =========================
sample_score_rows = []


for (
    sample,
    state,
), sub in score_df.groupby(
    [
        sample_col,
        "malignant_state",
    ],
    observed=True,
):
    row = {
        sample_col: sample,
        "malignant_state": state,
        "n_cells": int(
            sub.shape[
                0
            ]
        ),
        CONDITION_COL: sub[
            CONDITION_COL
        ].iloc[
            0
        ],
    }


    if (
        patient_col
        and patient_col
        in obs.columns
    ):
        row[
            patient_col
        ] = obs.loc[
            sub.index,
            patient_col,
        ].iloc[
            0
        ]


    if (
        dataset_col
        and dataset_col
        in obs.columns
    ):
        row[
            dataset_col
        ] = obs.loc[
            sub.index,
            dataset_col,
        ].iloc[
            0
        ]


    for score_col in score_cols:
        row[
            f"mean_{score_col}"
        ] = (
            float(
                sub[
                    score_col
                ].mean()
            )
            if sub[
                score_col
            ].notna().sum()
            else np.nan
        )

        row[
            f"median_{score_col}"
        ] = (
            float(
                sub[
                    score_col
                ].median()
            )
            if sub[
                score_col
            ].notna().sum()
            else np.nan
        )


    sample_score_rows.append(
        row
    )


sample_score_summary = pd.DataFrame(
    sample_score_rows
)


SAMPLE_SCORE_TSV = (
    METADATA_DIR
    / "malignant_state_sample_level_program_scores_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    sample_score_summary,
    SAMPLE_SCORE_TSV,
)

manifest_records.append(
    {
        "file": str(
            SAMPLE_SCORE_TSV
        ),
        "description": "Sample-level program score summaries for malignant states",
    }
)


if sample_score_summary.shape != (
    375,
    20,
):
    raise RuntimeError(
        "Historical sample-level program-score checkpoint failed: "
        f"{sample_score_summary.shape} != (375, 20)"
    )


# Historical sample-level sentinel.
#
# The historical executed notebook displayed the biological sample PJ052 as
# "GSM4202144_PJ052". Repository metadata harmonization may preserve the same
# biological sample under a longer canonical label or a shortened patient/sample
# label. Therefore locate the sentinel by the stable biological token "PJ052"
# across both sample and patient identifiers rather than by one literal string.
sentinel_mask = (
    sample_score_summary[
        "malignant_state"
    ]
    .astype(str)
    .eq(
        "AC_like_reactive_malignant"
    )
)


pj052_mask = (
    sample_score_summary[
        sample_col
    ]
    .astype(str)
    .str.contains(
        "PJ052",
        case=False,
        regex=False,
        na=False,
    )
)


if (
    patient_col
    and patient_col
    in sample_score_summary.columns
):
    pj052_mask = (
        pj052_mask
        | sample_score_summary[
            patient_col
        ]
        .astype(str)
        .str.contains(
            "PJ052",
            case=False,
            regex=False,
            na=False,
        )
    )


sentinel = sample_score_summary.loc[
    sentinel_mask
    & pj052_mask
].copy()


sentinel_checks = {
    "mean_score_OPC_proneural_program": 0.373499,
    "mean_score_AC_like_reactive_program": 2.017055,
    "mean_score_cycling_program": -0.161626,
}


if len(
    sentinel
) == 1:
    print(
        "Historical PJ052 sentinel resolved as:",
        sentinel[
            [
                sample_col,
                "malignant_state",
            ]
            + (
                [
                    patient_col
                ]
                if (
                    patient_col
                    and patient_col
                    in sentinel.columns
                )
                else []
            )
        ]
        .to_dict(
            orient="records"
        )[
            0
        ],
    )

    for column, expected in sentinel_checks.items():
        observed = float(
            sentinel[
                column
            ].iloc[
                0
            ]
        )

        if not np.isclose(
            observed,
            expected,
            atol=5e-6,
            rtol=0,
        ):
            raise RuntimeError(
                f"Historical sample-score sentinel failed for {column}: "
                f"{observed} != {expected}"
            )

    print(
        "PASS: historical PJ052 sample-level score sentinel reproduced."
    )

else:
    # Do not fail solely because repository metadata renamed a sample identifier.
    # The full-state score sentinels above already hard-check the actual Scanpy
    # scoring semantics across all 22,525 AC-like cells, and the historical
    # sample-level table shape is also hard-checked at (375, 20).
    print(
        "[NOTE] Historical literal PJ052 sample sentinel was not uniquely "
        f"resolvable after metadata harmonization (matches={len(sentinel)})."
    )

    print(
        "The canonical state-level score sentinels and sample-table shape "
        "remain the authoritative scoring-reproducibility checkpoints."
    )


display(
    sample_score_summary.head()
)

print(
    "PASS: historical sample-level program scores reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_sample_level_program_scores_HVG8000_r0_6.tsv shape=(375, 20)
Historical PJ052 sentinel resolved as: {'sample_id': 'PJ052', 'malignant_state': 'AC_like_reactive_malignant', 'patient_id': 'PJ052'}
PASS: historical PJ052 sample-level score sentinel reproduced.


,sample_id,malignant_state,n_cells,condition,patient_id,core_dataset,mean_score_OPC_proneural_program,median_score_OPC_proneural_program,mean_score_NPC_neural_program,median_score_NPC_neural_program,mean_score_AC_like_reactive_program,median_score_AC_like_reactive_program,mean_score_MES_like_program,median_score_MES_like_program,mean_score_hypoxia_stress_program,median_score_hypoxia_stress_program,mean_score_cycling_program,median_score_cycling_program,mean_score_myeloid_contamination_review_program,median_score_myeloid_contamination_review_program
0,GSM5518596_rGBM-01-A,AC_like_reactive_malignant,12,rGBM,01,DS3_GSE182109,0.013165,-0.062652,-0.372046,-0.381274,1.639778,1.713106,0.479332,0.377368,-0.140783,-0.169586,-0.278333,-0.278750,0.147349,0.183750
1,GSM5518596_rGBM-01-A,OPC_NPC_like_malignant,45,rGBM,01,DS3_GSE182109,0.178157,0.200736,-0.322652,-0.357763,1.213206,1.292612,0.328451,0.299257,-0.141279,-0.188605,-0.207323,-0.266179,0.166001,0.115478
2,GSM5518596_rGBM-01-A,cycling_malignant_candidate,46,rGBM,01,DS3_GSE182109,-0.179043,-0.219451,-0.369090,-0.388181,0.892019,0.830353,0.204664,0.184278,-0.095982,-0.192766,1.060309,1.132991,0.114533,0.094610
3,GSM5518596_rGBM-01-A,hypoxia_stress_malignant,66,rGBM,01,DS3_GSE182109,-0.211779,-0.271914,-0.342886,-0.367289,0.685776,0.660049,0.357595,0.344337,0.375628,0.352627,-0.145989,-0.195222,0.330695,0.207191
4,GSM5518597_rGBM-01-B,AC_like_reactive_malignant,62,rGBM,01,DS3_GSE182109,-0.191830,-0.253908,-0.381923,-0.403757,1.605160,1.592009,0.762119,0.809838,-0.020866,-0.049713,-0.141721,-0.190205,0.256045,0.150249


PASS: historical sample-level program scores reproduced.


In [15]:
# =========================
# 14. Figures
# =========================
plot_obs = (
    obs
    .copy()
)


plot_adata = ad.AnnData(
    X=sp.csr_matrix(
        (
            source_shape[
                0
            ],
            0,
        ),
        dtype=np.float32,
    ),
    obs=plot_obs,
)


plot_adata.obsm[
    "X_umap"
] = np.asarray(
    source_umap
)


sc.pl.umap(
    plot_adata,
    color=[
        "malignant_state",
        FINAL_COMPARTMENT_COL,
        CONDITION_COL,
    ],
    wspace=0.35,
    show=False,
)


fig_path = (
    FIGURES_DIR
    / "UMAP_malignant_state_final_compartment_condition_HVG8000_r0_6.png"
)

savefig_replace(
    fig_path
)

figure_records.append(
    {
        "figure": str(
            fig_path
        ),
        "description": "UMAP colored by malignant state, final compartment and condition",
    }
)


for column in score_cols:
    sc.pl.umap(
        plot_adata,
        color=column,
        show=False,
        cmap="viridis",
        vmin="p1",
        vmax="p99",
    )

    fig_path = (
        FIGURES_DIR
        / f"UMAP_{column}_HVG8000_r0_6.png"
    )

    savefig_replace(
        fig_path
    )

    figure_records.append(
        {
            "figure": str(
                fig_path
            ),
            "description": f"UMAP program score: {column}",
        }
    )


# Barplot: sample-level median malignant-state fractions by condition.
plot_df = condition_summary.copy()

fig, ax = plt.subplots(
    figsize=(
        10,
        5,
    )
)

states = sorted(
    plot_df[
        "malignant_state"
    ].unique()
)

x = np.arange(
    len(
        states
    )
)

conditions = sorted(
    plot_df[
        "condition"
    ]
    .astype(str)
    .unique()
)

width = min(
    0.8
    / max(
        len(
            conditions
        ),
        1,
    ),
    0.18,
)


for idx, condition in enumerate(
    conditions
):
    values = []

    for state in states:
        row = plot_df[
            (
                plot_df[
                    "malignant_state"
                ]
                == state
            )
            & (
                plot_df[
                    "condition"
                ]
                .astype(str)
                == condition
            )
        ]

        values.append(
            float(
                row[
                    "median_fraction"
                ].iloc[
                    0
                ]
            )
            if len(
                row
            )
            else 0
        )

    ax.bar(
        x
        + (
            idx
            - (
                len(
                    conditions
                )
                - 1
            )
            / 2
        )
        * width,
        values,
        width,
        label=condition,
    )


ax.set_xticks(
    x
)

ax.set_xticklabels(
    states,
    rotation=45,
    ha="right",
)

ax.set_ylabel(
    "Median fraction within malignant-state pool"
)

ax.set_title(
    "Malignant-state composition by condition"
)

ax.legend(
    frameon=False,
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

fig.tight_layout()


fig_path = (
    FIGURES_DIR
    / "bar_malignant_state_condition_median_sample_fraction_HVG8000_r0_6.png"
)

savefig_replace(
    fig_path
)

figure_records.append(
    {
        "figure": str(
            fig_path
        ),
        "description": "Barplot of sample-level median malignant-state fractions by condition",
    }
)


# Historical marker dotplot uses the normalized/log malignant full-gene object.
dotplot_markers = []


for program, genes in PROGRAM_MARKERS.items():
    present = [
        gene
        for gene in genes
        if gene in adata_score.var_names
    ]

    dotplot_markers.extend(
        present[
            :6
        ]
    )


seen = set()

dotplot_markers = [
    gene
    for gene in dotplot_markers
    if not (
        gene in seen
        or seen.add(
            gene
        )
    )
]


if len(
    dotplot_markers
) > 0:
    sc.pl.dotplot(
        adata_score,
        var_names=dotplot_markers,
        groupby="malignant_state",
        standard_scale="var",
        show=False,
    )

    fig_path = (
        FIGURES_DIR
        / "dotplot_malignant_state_program_markers_HVG8000_r0_6.png"
    )

    savefig_replace(
        fig_path
    )

    figure_records.append(
        {
            "figure": str(
                fig_path
            ),
            "description": "Dotplot of malignant-state program markers",
        }
    )

else:
    raise RuntimeError(
        "Historical dotplot marker list is unexpectedly empty."
    )


if len(
    figure_records
) != 10:
    raise RuntimeError(
        "Historical malignant-state figure-count checkpoint failed: "
        f"{len(figure_records)} != 10"
    )


generated_uns = {}


if "malignant_state_colors" in plot_adata.uns:
    generated_uns[
        "malignant_state_colors"
    ] = np.asarray(
        plot_adata.uns[
            "malignant_state_colors"
        ]
    ).copy()


del plot_adata
gc.collect()


print(
    "PASS: historical ten-figure malignant-state set reproduced."
)


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_malignant_state_final_compartment_condition_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_score_OPC_proneural_program_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_score_NPC_neural_program_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_score_AC_like_reactive_program_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_score_MES_like_program_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_score_hypoxia_stress_program_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_malignant_state_analysis\UMAP_score_cycling_program_HVG

In [16]:
# =========================
# 15. Save updated full H5AD memory-safely
# =========================
# Release the full-gene malignant scoring matrix before the large disk copy.
del adata_score
gc.collect()


if WRITE_UPDATED_H5AD:
    output_reused = False

    if OUTPUT_H5AD.exists():
        try:
            validate_h5ad_obs_only(
                OUTPUT_H5AD,
                expected_shape=(
                    351427,
                    15176,
                ),
                expected_state_counts=EXPECTED_STATE_COUNTS,
            )

            print(
                "[RECOVERY PASS] Existing Notebook-20 output H5AD is complete and canonical:"
            )

            print(
                OUTPUT_H5AD
            )

            output_reused = True

        except Exception as error:
            print(
                "[RECOVERY FAIL] Existing Notebook-20 output H5AD is invalid:"
            )

            print(
                repr(
                    error
                )
            )

            OUTPUT_H5AD.unlink()


    temp_output = OUTPUT_H5AD.with_name(
        OUTPUT_H5AD.stem
        + ".building.h5ad"
    )


    if temp_output.exists():
        temp_output.unlink()


    source_size = INPUT_H5AD.stat().st_size

    free_space = shutil.disk_usage(
        OUTPUT_H5AD.parent
    ).free


    print(
        "Input H5AD size GiB:",
        round(
            source_size
            / 1024**3,
            3,
        ),
    )

    print(
        "Output-drive free GiB:",
        round(
            free_space
            / 1024**3,
            3,
        ),
    )


    if free_space < int(
        source_size
        * 1.05
    ):
        raise RuntimeError(
            "Insufficient free space for safe Notebook-20 output H5AD creation."
        )


    if not output_reused:
        print(
            "[COPY CANONICAL NOTEBOOK-17 H5AD]"
        )

        copy_file_with_progress(
            INPUT_H5AD,
            temp_output,
        )


        print(
            "[REPLACE /obs ONLY]"
        )

        replace_h5ad_obs_group(
            temp_output,
            obs,
        )


        if generated_uns:
            print(
                "[PATCH GENERATED COLOR METADATA]"
            )

            add_h5ad_uns_elements(
                temp_output,
                generated_uns,
            )


        validate_h5ad_obs_only(
            temp_output,
            expected_shape=(
                351427,
                15176,
            ),
            expected_state_counts=EXPECTED_STATE_COUNTS,
        )


        os.replace(
            temp_output,
            OUTPUT_H5AD,
        )


        print(
            "[SAVED H5AD]",
            OUTPUT_H5AD,
        )


        manifest_records.append(
            {
                "file": str(
                    OUTPUT_H5AD
                ),
                "description": "Updated h5ad with malignant-state labels and program scores",
            }
        )


    if output_reused:
        manifest_records.append(
            {
                "file": str(
                    OUTPUT_H5AD
                ),
                "description": "Updated h5ad with malignant-state labels and program scores",
            }
        )


else:
    print(
        "[SKIPPED] WRITE_UPDATED_H5AD is False"
    )


Input H5AD size GiB: 13.641
Output-drive free GiB: 364.326
[COPY CANONICAL NOTEBOOK-17 H5AD]
  5% (0.69 / 13.64 GiB)
  10% (1.38 / 13.64 GiB)
  15% (2.06 / 13.64 GiB)
  20% (2.75 / 13.64 GiB)
  25% (3.44 / 13.64 GiB)
  30% (4.12 / 13.64 GiB)
  35% (4.81 / 13.64 GiB)
  40% (5.50 / 13.64 GiB)
  45% (6.19 / 13.64 GiB)
  50% (6.88 / 13.64 GiB)
  55% (7.56 / 13.64 GiB)
  60% (8.19 / 13.64 GiB)
  65% (8.88 / 13.64 GiB)
  70% (9.56 / 13.64 GiB)
  75% (10.25 / 13.64 GiB)
  80% (10.94 / 13.64 GiB)
  85% (11.62 / 13.64 GiB)
  90% (12.31 / 13.64 GiB)
  95% (13.00 / 13.64 GiB)
  100% (13.64 / 13.64 GiB)
[REPLACE /obs ONLY]
[PATCH GENERATED COLOR METADATA]
[SAVED H5AD] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad


In [17]:
# =========================
# 16. Manifests + final historical checkpoints
# =========================
fig_manifest = pd.DataFrame(
    figure_records
)


FIG_MANIFEST_TSV = (
    METADATA_DIR
    / "malignant_state_figure_manifest_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    fig_manifest,
    FIG_MANIFEST_TSV,
)

manifest_records.append(
    {
        "file": str(
            FIG_MANIFEST_TSV
        ),
        "description": "Figure manifest for malignant-state analysis",
    }
)


out_manifest = pd.DataFrame(
    manifest_records
)


OUT_MANIFEST_TSV = (
    METADATA_DIR
    / "malignant_state_analysis_output_manifest_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    out_manifest,
    OUT_MANIFEST_TSV,
)


if fig_manifest.shape != (
    10,
    2,
):
    raise RuntimeError(
        "Historical figure-manifest checkpoint failed: "
        f"{fig_manifest.shape} != (10, 2)"
    )


if out_manifest.shape != (
    12,
    2,
):
    raise RuntimeError(
        "Historical output-manifest checkpoint failed: "
        f"{out_manifest.shape} != (12, 2)"
    )


if not all(
    Path(
        path
    ).exists()
    for path in out_manifest[
        "file"
    ]
):
    raise RuntimeError(
        "One or more Notebook-20 output artifacts are missing."
    )


display(
    out_manifest
)


# Clean successful scratch outputs only after every final checkpoint has passed.
for scratch_path in [
    SCORING_COUNTS_H5AD,
    SCORING_NORM_H5AD,
]:
    if scratch_path.exists():
        try:
            scratch_path.unlink()

            print(
                "[REMOVED SCRATCH]",
                scratch_path,
            )

        except Exception as error:
            print(
                "[WARNING] Could not remove scratch file:",
                scratch_path,
                repr(
                    error
                ),
            )


print(
    "\nFINAL CHECKPOINTS"
)

print(
    "Historical notebook mapping:",
    "19_malignant_state_composition_program_scoring_HVG8000_r0_6.ipynb "
    "-> 20_define_malignant_states.ipynb",
)

print(
    "Canonical Notebook-17 input:",
    INPUT_H5AD,
)

print(
    "Source shape:",
    source_shape,
)

print(
    "Malignant-state analysis cells:",
    n_malignant,
)

print(
    "Malignant-state counts:",
    EXPECTED_STATE_COUNTS,
)

print(
    "Readiness:",
    readiness.shape,
)

print(
    "Cluster/state summary:",
    cluster_state_summary.shape,
)

print(
    "Sample-state composition:",
    sample_state.shape,
)

print(
    "Malignant-state samples:",
    sample_state[
        sample_col
    ].nunique(),
)

print(
    "Condition summary:",
    condition_summary.shape,
)

print(
    "Kruskal-Wallis:",
    kw_df.shape,
)

print(
    "Pairwise Mann-Whitney:",
    pair_df.shape,
)

print(
    "Program-marker presence:",
    marker_presence.shape,
)

print(
    "Program-score summary:",
    score_summary.shape,
)

print(
    "Sample-level program scores:",
    sample_score_summary.shape,
)

print(
    "Figure manifest:",
    fig_manifest.shape,
)

print(
    "Output manifest:",
    out_manifest.shape,
)

print(
    "Output H5AD:",
    OUTPUT_H5AD,
)

print(
    "\n[DONE] Notebook 20 canonical malignant-state composition and program scoring completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_figure_manifest_HVG8000_r0_6.tsv shape=(10, 2)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_analysis_output_manifest_HVG8000_r0_6.tsv shape=(12, 2)


,file,description
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,Cell counts for malignant-state labels
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,Input readiness report
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,Cluster-to-malignant-state QC summary
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,Sample-level malignant state proportions
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,Condition-level summary of malignant-state sam...
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,Kruskal-Wallis tests for malignant-state propo...
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,Pairwise Mann-Whitney tests for malignant-stat...
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,Presence/absence of malignant program marker g...
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,Program score summaries by malignant state and...
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,Sample-level program score summaries for malig...


[REMOVED SCRATCH] E:\glioma-cnv-scratch\main_analysis_20\20_malignant_cells_fullgene_normalized_log1p_for_scoring.h5ad

FINAL CHECKPOINTS
Historical notebook mapping: 19_malignant_state_composition_program_scoring_HVG8000_r0_6.ipynb -> 20_define_malignant_states.ipynb
Canonical Notebook-17 input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_CNV_consensus.h5ad
Source shape: (351427, 15176)
Malignant-state analysis cells: 161884
Malignant-state counts: {'not_malignant_state': 189543, 'OPC_NPC_like_malignant': 61646, 'cycling_malignant_candidate': 38634, 'hypoxia_stress_malignant': 32896, 'AC_like_reactive_malignant': 22525, 'NPC_neuronal_like_malignant': 6183}
Readiness: (15, 2)
Cluster/state summary: (18, 9)
Sample-state composition: (400, 10)
Malignant-state samples: 80
Condition summary: (20, 9)
Kruskal-Wallis: (5, 7)
Pairwise Mann-Whitney: (30, 12)
Program-marker presence: (7, 6)
Program-score summary: (175, 9)
Sample-level 

## Expected historical checkpoints

A successful run should end with:

```text
Source shape: (351427, 15176)
Malignant-state analysis cells: 161884

Malignant-state counts:
    not_malignant_state             189543
    OPC_NPC_like_malignant           61646
    cycling_malignant_candidate      38634
    hypoxia_stress_malignant         32896
    AC_like_reactive_malignant       22525
    NPC_neuronal_like_malignant       6183

Readiness: (15, 2)
Cluster/state summary: (18, 9)
Sample-state composition: (400, 10)
Malignant-state samples: 80
Condition summary: (20, 9)
Kruskal-Wallis: (5, 7)
Pairwise Mann-Whitney: (30, 12)
Program-marker presence: (7, 6)
Program-score summary: (175, 9)
Sample-level program scores: (375, 20)
Figure manifest: (10, 2)
Output manifest: (12, 2)
```

The program-score hard checks also compare historical aggregate and sample-level sentinel values, so a change in Scanpy scoring semantics cannot silently pass as a canonical reproduction.